In [ ]:
!pip install torch torchvision pandas scikit-learn pillow

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

import pandas as pd
from PIL import Image
import os

In [ ]:
class HousingDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.data = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tabular = torch.tensor(self.data.iloc[idx, 1:-1].values, dtype=torch.float32)

        img_path = os.path.join(self.img_dir, self.data.iloc[idx, 0])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        price = torch.tensor(self.data.iloc[idx, -1], dtype=torch.float32)

        return tabular, image, price

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [ ]:
import pandas as pd
df = pd.read_csv("Housing.csv")

In [1]:
from google.colab import files
uploaded = files.upload()

import zipfile
import io

with zipfile.ZipFile(io.BytesIO(uploaded['Housing.zip']), 'r') as zip_ref:
    zip_ref.extractall('data')

Saving Housing.zip to Housing.zip


In [1]:
import torch.nn as nn
import torchvision.models as models

class MultiModalModel(nn.Module):
    def __init__(self, tabular_dim):
        super().__init__()

        self.cnn = models.resnet18(pretrained=True)
        self.cnn.fc = nn.Linear(self.cnn.fc.in_features, 64)

        self.tabular_net = nn.Sequential(
            nn.Linear(tabular_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32)
        )

        self.fusion = nn.Sequential(
            nn.Linear(64 + 32, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, tabular, image):
        img_features = self.cnn(image)
        tab_features = self.tabular_net(tabular)

        combined = torch.cat((img_features, tab_features), dim=1)
        output = self.fusion(combined)

        return output

In [15]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

import pandas as pd
from PIL import Image
import os

from google.colab import files
import zipfile
import io

class HousingDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.full_data = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

        self.image_id_col = 'image_id'
        self.numerical_features = ['bed', 'bath', 'sqft']
        self.target_col = 'price'

        for col in self.numerical_features:
            self.full_data[col] = pd.to_numeric(self.full_data[col], errors='coerce')
        self.full_data[self.numerical_features] = self.full_data[self.numerical_features].fillna(0)

        self.tabular_data = self.full_data[self.numerical_features]
        self.image_filenames = self.full_data[self.image_id_col]
        self.prices = self.full_data[self.target_col]

    def __len__(self):
        return len(self.full_data)

    def __getitem__(self, idx):
        tabular = torch.tensor(self.tabular_data.iloc[idx].values, dtype=torch.float32)

        img_filename = str(self.image_filenames.iloc[idx])
        img_path = os.path.join(self.img_dir, img_filename)
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        price = torch.tensor(self.prices.iloc[idx], dtype=torch.float32)

        return tabular, image, price

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

if not os.path.exists('Housing.csv'):
    print("Please upload 'Housing.csv'")
    uploaded_csv = files.upload()
    if 'Housing.csv' not in uploaded_csv:
        raise FileNotFoundError("Housing.csv not uploaded.")

if not os.path.exists('data'):
    print("Please upload 'Housing.zip' (containing images) and it will be extracted to a 'data' directory.")
    uploaded_zip = files.upload()
    if 'Housing.zip' in uploaded_zip:
        with zipfile.ZipFile(io.BytesIO(uploaded_zip['Housing.zip']), 'r') as zip_ref:
            zip_ref.extractall('data')
    else:
        raise FileNotFoundError("Housing.zip not uploaded.")

dataset = HousingDataset(csv_file='Housing.csv', img_dir='data', transform=transform)

loader = DataLoader(dataset, batch_size=32, shuffle=True)

tabular_dim = len(dataset.numerical_features)
model = MultiModalModel(tabular_dim)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


MultiModalModel(
  (cnn): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track

In [10]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)